# 작업형 제2유형 — 기본 요점정리

**생성일:** 2026-05-05  
**목적:** ADP 작업형 제2유형(회귀/분류 예측) 핵심 패턴 정리

---

## A. 기본 세팅 (Import & 한글 폰트)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## B. 데이터 로드 패턴

In [ ]:
# 시험 환경 기본 로드 패턴
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(train.shape, test.shape)
print(train.info())
print(train.describe())

## C. 전처리 핵심 패턴

In [ ]:
# IQR 이상치 처리 (단일 컬럼 예시)
def iqr_clip(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    return df

# 결측치 처리
# train[col].fillna(train[col].median(), inplace=True)
# test[col].fillna(train[col].median(), inplace=True)  # train 통계 기준!

# 범주형 인코딩 (manual map 권장)
# mapping = {'A': 0, 'B': 1, 'C': 2}
# train['cat_col'] = train['cat_col'].map(mapping)
# test['cat_col']  = test['cat_col'].map(mapping)

## D. 모델 학습 — 회귀 (Regression)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 피처/타겟 분리
target = 'grade'  # 문제마다 다름
features = [c for c in train.columns if c != target]

X = train[features]
y = train[target]
X_test = test[features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- XGBRegressor (기본 선택) ---
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_val_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
print('Validation RMSE:', round(rmse, 2))

# 최종 예측 & 저장
y_test_pred = model.predict(X_test)
pd.DataFrame({'pred': y_test_pred}).to_csv('result.csv', index=False)
print('result.csv 저장 완료')

## E. 모델 학습 — 분류 (Classification)

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score
from xgboost import XGBClassifier

model_clf = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
model_clf.fit(X_train, y_train)

y_val_prob = model_clf.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_val_prob)
print('Validation AUC:', round(auc, 4))

# 최종 예측 & 저장
y_test_prob = model_clf.predict_proba(X_test)[:, 1]
pd.DataFrame({'pred': y_test_prob}).to_csv('result.csv', index=False)
print('result.csv 저장 완료')

## F. result.csv 저장 형식 (반드시 준수)

In [ ]:
# 회귀: pred 컬럼에 예측값
pd.DataFrame({'pred': y_test_pred}).to_csv('result.csv', index=False)

# 분류(이진): pred 컬럼에 확률값
pd.DataFrame({'pred': y_test_prob}).to_csv('result.csv', index=False)

# 확인
pd.read_csv('result.csv').head()

---

## 오답 보강 섹션

> 이 섹션은 오답 분석 에이전트로부터 라우팅된 개념 보강 내용입니다.

### ⚠️ 오답 보강 — 회귀 모델 선택 (LinearRegression vs XGBRegressor/RandomForestRegressor)

- **발생 오류 유형:** CONCEPT_ERROR
- **틀린 이유:** LinearRegression은 선형 관계만 포착하여 복잡한 비선형 패턴이 있는 데이터에서 RMSE가 높음. ADP 시험에서 회귀 문제는 XGBRegressor 또는 RandomForestRegressor가 높은 점수를 받기 쉬움.

In [ ]:
# ❌ 틀린 코드 (RMSE 높음)
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(round(rmse, 2))  # 288.22

In [ ]:
# ✅ 올바른 코드 — XGBRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(round(rmse, 2))

In [ ]:
# ✅ 올바른 코드 — RandomForestRegressor (대안)
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(round(rmse, 2))

> 💡 **시험 팁:** ADP 회귀 문제는 특별한 언급이 없으면 `XGBRegressor(n_estimators=100, random_state=42)`를 기본 선택으로 사용하라. LinearRegression은 베이스라인 확인용으로만 쓰고 최종 제출 모델로는 사용하지 말 것.